# Benchmark de busca semantica em patentes - pipeline completoGera embeddings com **EmbeddingGemma-300M**, roda a busca por similaridade eavalia com nDCG, MRR, Recall e Precisao, comparando as tres variantes de textodo corpus (1.000 patentes do INPI, 45 queries em 3 temas).**Antes de rodar:**1. `Ambiente de execucao > Alterar o tipo de ambiente de execucao > GPU (T4)`.2. O modelo `google/embeddinggemma-300m` e **gated** no Hugging Face. Cada pessoa   precisa, na propria conta: aceitar a licenca em   https://huggingface.co/google/embeddinggemma-300m e criar um token em   https://huggingface.co/settings/tokens.3. Guarde o token nos **Secrets do Colab** (icone de chave na barra lateral) com   o nome `HF_TOKEN` e ative o acesso para este notebook. Secrets nao sao   compartilhados entre contas: seu colega precisa cadastrar o token dele.O pipeline inteiro roda em poucos minutos numa T4.

## 1. Ambiente

In [ ]:
!pip -q install -U "sentence-transformers>=5.0.0" pandas numpy

In [ ]:
import torchprint("GPU disponivel:", torch.cuda.is_available())if torch.cuda.is_available():    print("Dispositivo:", torch.cuda.get_device_name(0))else:    print("AVISO: sem GPU. Vai rodar em CPU (mais lento, mas viavel para 1.000 docs).")    print("Ative em: Ambiente de execucao > Alterar o tipo de ambiente de execucao > GPU")

### Autenticacao no Hugging Face

In [ ]:
import osfrom huggingface_hub import logintoken = Nonetry:    from google.colab import userdata    token = userdata.get("HF_TOKEN")except Exception:    token = os.environ.get("HF_TOKEN")if not token:    from getpass import getpass    token = getpass("Cole seu token do Hugging Face: ")login(token=token)print("Autenticado no Hugging Face.")

## 2. Codigo e dadosAjuste `REPO_URL` para o endereco do repositorio. Se o repositorio for privado,use a forma `https://<usuario>:<token_github>@github.com/<usuario>/<repo>.git`ou configure uma chave SSH.

In [ ]:
from pathlib import Pathimport sysREPO_URL = "https://github.com/SEU_USUARIO/benchmark_patentes_semantica.git"REPO_DIR = Path("/content/benchmark_patentes_semantica")if REPO_DIR.exists():    !cd {REPO_DIR} && git pull --ff-onlyelse:    !git clone {REPO_URL} {REPO_DIR}sys.path.insert(0, str(REPO_DIR))print(sorted(p.name for p in REPO_DIR.glob("*.tsv")))

### Onde salvar os embeddingsPor padrao os embeddings vao para o disco efemero do Colab (`/content`), que eapagado quando a sessao encerra. Sao ~3 MB por colecao e a geracao leva poucosminutos, entao regerar e barato.Se preferir persistir entre sessoes, defina `USAR_DRIVE = True` — util para naoregerar a cada vez, e para que os dois vejam exatamente os mesmos vetores.

In [ ]:
USAR_DRIVE = Falseif USAR_DRIVE:    from google.colab import drive    drive.mount("/content/drive")    SAIDA_DIR = Path("/content/drive/MyDrive/benchmark_patentes/embeddings")else:    SAIDA_DIR = REPO_DIR / "embeddings"SAIDA_DIR.mkdir(parents=True, exist_ok=True)print("Embeddings em:", SAIDA_DIR)

## 3. ConfiguracaoAs tres variantes de documento diferem apenas na coluna de texto usada:| variante | conteudo do texto ||---|---|| `tr` | titulo + resumo || `ipc_direto` | titulo + resumo + descricoes IPC em PT || `ipc_hierarquia` | titulo + resumo + hierarquia IPC completa em PT |**Sobre os prompts:** o EmbeddingGemma tem prompts nativos distintos paradocumento e consulta (`encode_document` / `encode_query`). Usar os promptsnativos e o caminho recomendado — misturar prompt nativo de um lado cominstrucao manual do outro desalinha os vetores. O toggle`USAR_PROMPTS_NATIVOS = False` reproduz a abordagem de instrucao em PT doscript local, para quem quiser comparar as duas.

In [ ]:
import csvimport jsonimport timeimport gcimport numpy as npimport pandas as pdfrom sentence_transformers import SentenceTransformerMODEL_NAME = "google/embeddinggemma-300m"DIMENSAO_ESPERADA = 768MAX_SEQ_LENGTH = 2048BATCH_SIZE = 16BLOCK_SIZE = 500USAR_PROMPTS_NATIVOS = TrueDOC_INSTRUCTION = (    "Represente este documento de patente para busca semantica. "    "Foque no problema tecnico, na solucao proposta, no dominio de aplicacao "    "e na finalidade da invencao:")QUERY_INSTRUCTION = (    "Represente esta consulta de busca de patentes para recuperar documentos "    "tecnicamente relevantes:")DOC_VARIANTS = {    "tr": {        "input": REPO_DIR / "patentes_benchmark_amostra_1000.tsv",        "text_column": "texto_para_embedding",        "output_name": "gemma300_tr_docs",    },    "ipc_direto": {        "input": REPO_DIR / "patentes_benchmark_amostra_1000_ipc_pt.tsv",        "text_column": "texto_para_embedding_ipc_pt",        "output_name": "gemma300_tr_ipc_direto_pt_docs",    },    "ipc_hierarquia": {        "input": REPO_DIR / "patentes_benchmark_amostra_1000_ipc_pt.tsv",        "text_column": "texto_para_embedding_ipc_hierarquia_pt",        "output_name": "gemma300_tr_ipc_hierarquia_pt_docs",    },}QUERY_CONFIG = {    "input": REPO_DIR / "queries_benchmark_patentes.tsv",    "text_column": "query_text",    "output_name": "gemma300_queries",}CHAVE_DOC = "num_pedido_normalizado"CHAVE_QUERY = "query_id"

## 4. Funcoes de geracao

In [ ]:
def clean_text(valor):    if valor is None:        return ""    if isinstance(valor, float) and pd.isna(valor):        return ""    texto = str(valor)    if texto == "\\N":        return ""    return " ".join(texto.split()).strip()def carregar_tsv(path, text_column, limit=None):    df = pd.read_csv(        path, sep="\t", dtype=str, low_memory=False,        quoting=csv.QUOTE_NONE, escapechar="\\",    )    if text_column not in df.columns:        raise ValueError(f"Coluna {text_column} ausente em {path}")    if limit:        df = df.head(limit).copy()    df[text_column] = df[text_column].map(clean_text)    vazios = int((df[text_column] == "").sum())    if vazios:        print(f"  aviso: {vazios} textos vazios serao substituidos pelo titulo")        if "titulo" in df.columns:            faltantes = df[text_column] == ""            df.loc[faltantes, text_column] = df.loc[faltantes, "titulo"].map(clean_text)    return dfdef codificar(model, textos, kind):    """Codifica com fallback automatico de batch em caso de falta de memoria."""    batch = BATCH_SIZE    while True:        try:            if USAR_PROMPTS_NATIVOS:                fn = model.encode_document if kind == "docs" else model.encode_query                return fn(textos, batch_size=batch, show_progress_bar=True,                          convert_to_numpy=True, normalize_embeddings=True)            instrucao = DOC_INSTRUCTION if kind == "docs" else QUERY_INSTRUCTION            preparados = [f"{instrucao}\n\n{t}".strip() for t in textos]            return model.encode(preparados, batch_size=batch, show_progress_bar=True,                                convert_to_numpy=True, normalize_embeddings=True)        except Exception as exc:            memoria = isinstance(exc, MemoryError) or "out of memory" in str(exc).lower()            if not memoria or batch == 1:                raise            batch = max(1, batch // 2)            print(f"  memoria insuficiente; reduzindo batch para {batch}")            gc.collect()            if torch.cuda.is_available():                torch.cuda.empty_cache()def gerar_colecao(model, config, kind, chave, limit=None, overwrite=False):    saida = SAIDA_DIR / config["output_name"]    if saida.exists() and not overwrite and list(saida.glob("embeddings_bloco_*.npy")):        print(f"[{config['output_name']}] ja existe; pulando (use overwrite=True para regerar)")        return saida    saida.mkdir(parents=True, exist_ok=True)    df = carregar_tsv(config["input"], config["text_column"], limit)    textos = df[config["text_column"]].tolist()    print(f"[{config['output_name']}] {len(textos)} textos")    manifest = saida / "manifest.jsonl"    manifest.write_text("", encoding="utf-8")    inicio_geral = time.time()    for indice, comeco in enumerate(range(0, len(textos), BLOCK_SIZE)):        fatia = textos[comeco:comeco + BLOCK_SIZE]        vetores = np.asarray(codificar(model, fatia, kind), dtype=np.float32)        if vetores.shape != (len(fatia), DIMENSAO_ESPERADA):            raise ValueError(f"Shape inesperado: {vetores.shape}")        np.save(saida / f"embeddings_bloco_{indice:05d}.npy", vetores)        colunas = [c for c in (chave, "titulo", "ipc", "tema", "tipo_query", "query_text")                   if c in df.columns]        df.iloc[comeco:comeco + len(fatia)][colunas].to_csv(            saida / f"metadata_bloco_{indice:05d}.tsv", sep="\t",            index=False, quoting=csv.QUOTE_NONE, escapechar="\\",        )        with manifest.open("a", encoding="utf-8") as arq:            arq.write(json.dumps({                "indice_bloco": indice, "inicio": comeco,                "fim": comeco + len(fatia) - 1, "shape": list(vetores.shape),            }, ensure_ascii=False) + "\n")        print(f"  bloco {indice:05d}: {vetores.shape}")        del vetores        gc.collect()    (saida / "config.json").write_text(json.dumps({        "modelo": MODEL_NAME, "kind": kind,        "input": str(config["input"]), "text_column": config["text_column"],        "prompts_nativos": USAR_PROMPTS_NATIVOS,        "max_seq_length": MAX_SEQ_LENGTH, "batch_size": BATCH_SIZE,        "block_size": BLOCK_SIZE, "limit": limit,        "created_at": time.strftime("%Y-%m-%d %H:%M:%S"),        "duracao_s": round(time.time() - inicio_geral, 1),    }, ensure_ascii=False, indent=2), encoding="utf-8")    print(f"[{config['output_name']}] concluido em {time.time() - inicio_geral:.1f}s")    return saida

## 5. Carregar o modelo

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"model = SentenceTransformer(MODEL_NAME, device=device)model.max_seq_length = MAX_SEQ_LENGTHprint(f"Modelo carregado em {device} | dimensao {model.get_sentence_embedding_dimension()}")

### Smoke test (opcional, recomendado na primeira vez)Roda 50 documentos para confirmar que modelo, token e dados estao ok antes degastar os minutos da geracao completa.

In [ ]:
SMOKE = Trueif SMOKE:    gerar_colecao(        model,        {**DOC_VARIANTS["tr"], "output_name": "smoke50_tr_docs"},        kind="docs", chave=CHAVE_DOC, limit=50, overwrite=True,    )

## 6. Gerar as colecoes completas

In [ ]:
for nome, config in DOC_VARIANTS.items():    gerar_colecao(model, config, kind="docs", chave=CHAVE_DOC)gerar_colecao(model, QUERY_CONFIG, kind="queries", chave=CHAVE_QUERY)

## 7. Busca e avaliacaoA busca e uma multiplicacao de matrizes: com os vetores normalizados, o produtointerno **e** a similaridade de cosseno. Para 45 queries x 1.000 documentos issoleva milissegundos — nao ha necessidade de FAISS nem de indice aproximado nessaescala.

In [ ]:
import importlibimport avaliar_benchmark as abimportlib.reload(ab)qrels = ab.carregar_qrels(REPO_DIR / "qrels_candidatos_queries_benchmark.tsv")queries_df = pd.read_csv(REPO_DIR / "queries_benchmark_patentes.tsv", sep="\t", dtype=str)emb_queries = ab.carregar_colecao(SAIDA_DIR / QUERY_CONFIG["output_name"], CHAVE_QUERY)print(f"qrels: {len(qrels)} julgamentos | {qrels['query_id'].nunique()} queries")print("distribuicao de relevancia:")print(qrels["relevance"].value_counts().sort_index().to_string())

In [ ]:
rankings = {}resultados = {}for nome, config in DOC_VARIANTS.items():    emb_docs = ab.carregar_colecao(SAIDA_DIR / config["output_name"], CHAVE_DOC)    rankings[nome] = ab.buscar(emb_queries, emb_docs, top_k=100)    resultados[nome] = ab.avaliar(rankings[nome], qrels)    print(f"{nome}: avaliado sobre {len(resultados[nome]['por_query'])} queries")comparacao = ab.comparar_variantes(resultados)comparacao

### Quebra por tema e por tipo de queryOnde o modelo vai bem e onde vai mal costuma ser mais informativo que a mediageral — os tres temas (cancer/farmacos, 5G, purificacao de agua) temvocabularios muito diferentes, e os tipos de query (curta, tecnica, linguagemnatural) testam capacidades distintas.

In [ ]:
MELHOR = comparacao["nDCG@10"].idxmax()print(f"Melhor variante por nDCG@10: {MELHOR}\n")print("Por tema:")display(ab.avaliar_por_faceta(resultados[MELHOR], queries_df, "tema"))print("\nPor tipo de query:")display(ab.avaliar_por_faceta(resultados[MELHOR], queries_df, "tipo_query"))

### Inspecao qualitativaA tabela de metricas nao mostra *por que* uma query falhou. Olhar os documentosrecuperados e o passo que costuma revelar problemas no gabarito — inclusiveacertos do modelo que a regra marcou como irrelevantes.

In [ ]:
QUERY_INSPECIONAR = "QF003"emb_docs_melhor = ab.carregar_colecao(SAIDA_DIR / DOC_VARIANTS[MELHOR]["output_name"], CHAVE_DOC)docs_meta = pd.read_csv(    REPO_DIR / "patentes_benchmark_amostra_1000.tsv", sep="\t",    dtype=str, low_memory=False, quoting=csv.QUOTE_NONE, escapechar="\\",)texto_query = queries_df.loc[queries_df["query_id"] == QUERY_INSPECIONAR, "query_text"].iloc[0]print(f"{QUERY_INSPECIONAR}: {texto_query}\n")ab.inspecionar_query(rankings[MELHOR], QUERY_INSPECIONAR, docs_meta, qrels, top_n=10)

### Queries mais dificeisOrdenadas por nDCG@10 crescente: as primeiras sao as candidatas naturais pararevisao manual do gabarito.

In [ ]:
piores = (    resultados[MELHOR]["por_query"]    .merge(queries_df[["query_id", "tema", "tipo_query", "query_text"]], on="query_id")    .sort_values("nDCG@10")    [["query_id", "tema", "tipo_query", "nDCG@10", "MRR", "Recall@10", "query_text"]]    .head(10))piores

## 8. Salvar resultados

In [ ]:
RESULTADOS_DIR = REPO_DIR / "resultados"RESULTADOS_DIR.mkdir(exist_ok=True)carimbo = time.strftime("%Y%m%d_%H%M")comparacao.to_csv(RESULTADOS_DIR / f"comparacao_variantes_{carimbo}.csv")for nome, resultado in resultados.items():    resultado["por_query"].to_csv(        RESULTADOS_DIR / f"metricas_por_query_{nome}_{carimbo}.csv", index=False    )# top-20 de cada query da melhor variante, para revisao manual do gabaritolinhas = []for i, query_id in enumerate(rankings[MELHOR]["query_ids"]):    for posicao in range(20):        doc_id = str(rankings[MELHOR]["doc_ids_ranking"][i][posicao])        linhas.append({            "query_id": str(query_id),            "posicao": posicao + 1,            "num_pedido_normalizado": doc_id,            "score": float(rankings[MELHOR]["scores_ranking"][i][posicao]),        })top20 = pd.DataFrame(linhas).merge(    qrels.rename(columns={"relevance": "relevance_regra"}),    on=["query_id", "num_pedido_normalizado"], how="left",).merge(    docs_meta[["num_pedido_normalizado", "titulo", "ipc"]],    on="num_pedido_normalizado", how="left",)top20.to_csv(RESULTADOS_DIR / f"top20_para_revisao_{MELHOR}_{carimbo}.tsv", sep="\t", index=False)print("Arquivos salvos em", RESULTADOS_DIR)for arq in sorted(RESULTADOS_DIR.glob(f"*{carimbo}*")):    print(" -", arq.name)

## Como interpretar estes numerosO `qrels` atual foi gerado por regras (padroes de IPC + termos), e o proprioscript que o produziu se descreve como "intencionalmente fraco". Isso significaque as metricas acima medem **o quanto o modelo concorda com a regra**, nao oquanto ele acerta semanticamente. Um nDCG baixo pode indicar tanto uma falha domodelo quanto um acerto que a regra nao previu.O arquivo `top20_para_revisao_*.tsv` gerado acima serve exatamente para isso:comparar o que o modelo trouxe no topo com o rotulo da regra e corrigir ogabarito onde ele estiver errado. Depois da revisao, basta apontar`ab.carregar_qrels()` para o gabarito revisado e reexecutar a secao 7 — nenhumembedding precisa ser regerado.**Para trabalhar em dupla sem conflito:** cada um roda a propria copia destenotebook no Colab; as mudancas voltam ao GitHub por commit, nunca por edicaosimultanea do mesmo arquivo no Drive. Os arquivos em `resultados/` levamcarimbo de data e hora justamente para que duas execucoes nao se sobrescrevam.